This notebook explores route connectivity, network structure, and geographic insights from the OpenFlights dataset using DuckDB and Plotly.

In [2]:
import duckdb
import pandas as pd
import plotly.express as px
from pathlib import Path

repo_root = Path('..')
db_path = repo_root / 'data' / 'analytics.duckdb'
con = duckdb.connect(str(db_path))


This notebook uses curated dbt mart outputs to analyze route connectivity and airport hub structure. The visualizations are designed to show where the strongest international corridors and busiest airports are located, and which airlines dominate route coverage.

In [3]:
country_pairs = con.execute('''
select source_country, destination_country, route_count
from main.mart_top_country_pairs
order by route_count desc
limit 30
''').df()
country_pairs

,source_country,destination_country,route_count
0,United States,United States,10518
1,China,China,6877
2,Brazil,Brazil,1195
3,Canada,Canada,1167
4,India,India,1057
5,Russia,Russia,964
6,Australia,Australia,776
7,Japan,Japan,623
8,Indonesia,Indonesia,611
9,Spain,Spain,579


The sunburst chart below visualizes the most active international flight corridors by grouping routes first by origin country and then by destination country. Larger segments indicate heavier route volume, helping identify the strongest country-to-country connections.

In [4]:
fig = px.sunburst(
    country_pairs,
    path=['source_country', 'destination_country'],
    values='route_count',
    title='Top Country-to-Country Route Flows'
)
fig.show()


The geographic scatter plot maps the busiest hub airports and scales each marker by route volume. This makes it easy to see where major air connectivity is concentrated and which airports anchor the global network.

In [7]:
hub_airports = con.execute('''
select a.name as airport_name,
       a.city,
       a.country,
       a.latitude,
       a.longitude,
       total_routes
from main.mart_top_hub_airports h
join main.stg_airports a on a.airport_id = h.airport_id
order by total_routes desc
limit 50
''').df()

fig = px.scatter_geo(
    hub_airports,
    lat='latitude',
    lon='longitude',
    hover_name='airport_name',
    hover_data=['city', 'country', 'total_routes'],
    size='total_routes',
    projection='natural earth',
    title='Top Hub Airports by Route Activity'
)
fig.show()


The final bar chart ranks the top 20 airlines by route count. This allows us to compare carrier network reach, identify which airlines dominate route coverage, and understand how airline capacity is distributed across the network.

In [8]:
route_counts = con.execute('select * from main.mart_top_airlines_by_routes').df()
fig = px.bar(route_counts.head(20), x='airline', y='route_count', title='Top 20 Airlines by Route Count')
fig.show()
con.close()

Together, these charts show the structure of the OpenFlights network: the most connected country corridors, the geographic anchors of hub airports, and the airlines that drive the largest portion of route activity.